# Hybrid Quantum GNNs for Molecular Toxicity Classification
**BMS Institute of Technology and Management** | ML – BCS602 | AY 2025–2026  
**Guide:** Dr. Nagabhushan SV

| USN | Name |
|---|---|
| 1BY23CS014 | Aishwarya J A |
| 1BY23CS026 | Anurag Rai |
| 1BY23CS053 | Dasiga Venkata Ashish Kumar |
| 1BY23CS068 | G Nithish |

## 1. Install Dependencies

In [1]:
!pip install torch torch-geometric rdkit deepchem pennylane scikit-learn pandas tqdm matplotlib seaborn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 25.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 85.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 106.4 MB/s eta 0:00:0000:0100:01


## 3. Clone Repository (Colab only)

In [ ]:
try:
    import google.colab
    # Clone repo if in Colab
    !git clone https://github.com/r-anurag/Hybrid-Quantum-GNNs-for-Molecular-Toxicity-Classification.git
    %cd Hybrid-Quantum-GNNs-for-Molecular-Toxicity-Classification
    print('Repository cloned successfully')
except:
    print('Running locally, skipping clone')

## 4. Setup

In [3]:
import os, sys

# Detect environment and set paths
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Change to src directory
if os.path.exists('src'):
    os.chdir('src')
elif os.path.exists('../src'):
    os.chdir('../src')
else:
    print('ERROR: src/ folder not found')
    print(f'Current directory: {os.getcwd()}')
    print(f'Contents: {os.listdir()}')

import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Working directory: {os.getcwd()}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
Device: cpu


## 5. Setup Checkpoint Directory

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/HQGNN_checkpoints'
    print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
except:
    CHECKPOINT_DIR = '../checkpoints'
    print(f"Running locally, checkpoints saved to: {CHECKPOINT_DIR}")

## 6. Load Datasets

In [4]:
from data_pipeline import load_dataset

datasets = {}
for name in ("clintox", "tox21"):
    data_list, class_weights, tasks = load_dataset(name)
    datasets[name] = (data_list, class_weights, tasks)
    print(f"{name.upper()}: {len(data_list)} molecules, {len(tasks)} tasks")
    print(f"  Tasks: {tasks}")
    print(f"  Node features: {data_list[0].x.shape[1]}")

ModuleNotFoundError: No module named 'data_pipeline'

## 7. Define Model Variants

In [ ]:
from models import GCN, HybridQGNN, QuantumOnly

IN_CHANNELS = 10
EPOCHS      = 30
LR          = 1e-3
BATCH       = 64
N_FOLDS     = 5
# DEVICE is already set in cell 3

def get_variants(num_tasks):
    return {
        "Classical GCN": lambda: GCN(IN_CHANNELS, hidden=64, embed_dim=32, num_tasks=num_tasks),
        "Quantum-only 4-qubit": lambda: QuantumOnly(IN_CHANNELS, n_qubits=4, n_layers=2, num_tasks=num_tasks),
        "Hybrid 4-qubit": lambda: HybridQGNN(IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                                              n_qubits=4, n_layers=2, num_tasks=num_tasks),
        "Hybrid 8-qubit": lambda: HybridQGNN(IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                                              n_qubits=8, n_layers=2, num_tasks=num_tasks),
        "Hybrid 4-qubit + Edge": lambda: HybridQGNN(IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                                                     n_qubits=4, n_layers=2,
                                                     num_tasks=num_tasks, edge_embed=True),
    }

print("Model variants defined.")

## 8. Run Experiments

In [ ]:
from evaluate import cross_validate

all_rows = []

for ds_name, (data_list, class_weights, tasks) in datasets.items():
    num_tasks = len(tasks)
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name.upper()}  ({len(data_list)} molecules, {num_tasks} tasks)")
    print(f"{'='*60}")

    for name, model_fn in get_variants(num_tasks).items():
        print(f"\n>> {name}")
        n_params = sum(p.numel() for p in model_fn().parameters() if p.requires_grad)

        results = cross_validate(
            model_fn, data_list, class_weights,
            n_splits=N_FOLDS, epochs=EPOCHS, lr=LR,
            batch_size=BATCH, device=DEVICE, verbose=False,
            checkpoint_dir=CHECKPOINT_DIR, model_name=name.replace(" ", "_"),
            dataset_name=ds_name
        )
        row = {"Model": name, "Dataset": ds_name, "Params": n_params}
        row.update({k: v for k, v in results.items() if k != "histories"})
        all_rows.append({**row, "histories": results["histories"]})
        print(f"  AUC={results['auc_mean']:.4f}+/-{results['auc_std']:.4f}  "
              f"F1={results['f1_mean']:.4f}+/-{results['f1_std']:.4f}  "
              f"time/epoch={results['time_mean']:.2f}s")

df = pd.DataFrame([{k: v for k, v in r.items() if k != "histories"} for r in all_rows])
os.makedirs("../results", exist_ok=True)
df.to_csv("../results/results.csv", index=False)
print("\nResults saved to ../results/results.csv")

## 9. Results Table

In [ ]:
display(df[[
    "Dataset", "Model", "Params",
    "auc_mean", "auc_std",
    "f1_mean",  "f1_std",
    "time_mean"
]].round(4).style.background_gradient(subset=["auc_mean", "f1_mean"], cmap="YlGn"))

## 10. Performance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, label in zip(axes, ["auc_mean", "f1_mean"], ["ROC-AUC", "F1-Score"]):
    for ds_name in ("clintox", "tox21"):
        sub = df[df["Dataset"] == ds_name]
        ax.bar(
            [f"{m}\n({ds_name})" for m in sub["Model"]],
            sub[metric],
            yerr=sub[metric.replace("mean", "std")],
            capsize=4, label=ds_name
        )
    ax.set_title(label)
    ax.set_ylim(0, 1)
    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../results/performance_comparison.png", dpi=150)
plt.show()

## 11. Parameter Count Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
models_unique = df.drop_duplicates(subset="Model")
ax.barh(models_unique["Model"], models_unique["Params"], color="steelblue")
ax.set_xlabel("Parameter Count")
ax.set_title("Model Parameter Comparison")
plt.tight_layout()
plt.savefig("../results/parameter_comparison.png", dpi=150)
plt.show()

## 12. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, row in enumerate(all_rows[:6]):  # First 6 model-dataset combos
    ax = axes[idx]
    histories = row["histories"]
    
    # Average across folds
    train_losses = np.mean([h["train_loss"] for h in histories], axis=0)
    val_losses = np.mean([h["val_loss"] for h in histories], axis=0)
    
    ax.plot(train_losses, label="Train", linewidth=2)
    ax.plot(val_losses, label="Val", linewidth=2)
    ax.set_title(f"{row['Model']} - {row['Dataset'].upper()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/training_curves.png", dpi=150)
plt.show()